## Consumer Sentiment vs Consumer Consumption

### Preparation
---

#### Import statements

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import numpy as np
import scipy.stats as stats



pd.options.display.float_format ="{:,.2f}".format

#### Load the Datasets

In [5]:
consumer_sentiment_df = pd.read_csv("./Datasets/UMCSENT.csv")
consumer_economic_conditions_df= pd.read_excel("./Datasets/Current_Economic_Conditions.xlsx")
consumer_expectations_df= pd.read_excel("./Datasets/Consumer_Expectations.xlsx")
personal_consumption_exp_unadjusted = pd.read_csv("./Datasets/PCECA.csv")

FileNotFoundError: [Errno 2] No such file or directory: './Datasets/PCECA.csv'

### Initial checks
---

Use of the data is from 2000 onwards - so go to excel and remove the earlier values for ease.

<section class="data-checks">

  <h5>First Set of Checks</h5>
  <ul>
    <li>Head</li>
    <li>Tail</li>
    <li>Datatypes</li>
    <li>Columns</li>
    <li>Shape</li>
  </ul>

  <h5>Second Set of Checks</h5>
  <ul>
    <li>Remove unused columns</li>
    <li>Check duplicate values</li>
    <li>Missing values</li>
  </ul>
</section>


#### Consumer sentiment checks (Overall Sentiment)

In [ ]:
#### first set of checks
consumer_sentiment_df.sample(5)         # Date is not the index - np
consumer_sentiment_df.head()            # Starts at the correct date
consumer_sentiment_df.tail()            # ends at 2026-01
consumer_sentiment_df.columns           # No spaces which is good but do need to rename  + No need to drop columns
consumer_sentiment_df.dtypes            # Observation date needs to be converted

    #### Initial adjustments
consumer_sentiment_df.rename(columns={"observation_date":"Date","UMCSENT":"Sentiment_index"}, inplace=True)     # Fixed the index
consumer_sentiment_df["Date"]= pd.to_datetime(consumer_sentiment_df["Date"], format ="%d/%m/%Y")               # Datetime conversion but it is alsop in daily format so convert to thrifst of the mont



#### Second set of checks
consumer_sentiment_df.columns           #No need to drop columns
consumer_sentiment_df.duplicated(subset="Date", keep=False).value_counts()      # All are false so no duplicated values
consumer_sentiment_df.isna().sum()                                              # No missing values for any of the dates + Need to check if all the 12 months are there in the dataset

        # Checks to make sure that all the dates are there for the period from 2000 to 2026.
print(pd.date_range(start= "2000-01-01", end = "2026-01-01", freq="MS").difference(consumer_sentiment_df["Date"]))
                    # Sets up a date range, looks at the month start, then compares and then prints the ones that are different
                    ## There are no missing values

consumer_sentiment_df

#### Consumer Expectations checks

In [ ]:
#### first set of checks
consumer_expectations_df.sample(5)
consumer_expectations_df.head(30)       # Starts at the correct date
consumer_expectations_df.tail()         # ends at 2026-01
consumer_expectations_df.dtypes          # Observation no need to be converted since its a datetime.
consumer_expectations_df.columns        # No spaces which is good but do need to rename  + No need to drop columns

    #### Initial adjustments
consumer_expectations_df.rename(columns={"Datemy":"Date","ICE":"Consumer_Expectations_index"}, inplace=True)     # Fixed the index

#### Second set of checks
consumer_expectations_df.columns           #Need to remove the recessions column

consumer_expectations_df.duplicated(subset="Date", keep=False).value_counts()      # All are false so no duplicated values
consumer_expectations_df.isna().sum()                                              # No missing values for any of the dates + Need to check if all the 12 months are there in the dataset

        # Checks to make sure that all the dates are there for the period from 2000 to 2026.
print(pd.date_range(start= "2000-01-01", end = "2026-01-01", freq="MS").difference(consumer_expectations_df["Date"]))
                    # Sets up a date range, looks at the month start, then compares and then prints the ones that are different
                    # There are no missing values( from geeks for geeeks)

    #### Adjustments
consumer_expectations_df.drop(columns=["RECESSION"], inplace=True)          ## Remove the recession column
consumer_expectations_df["Date"]= pd.to_datetime(consumer_expectations_df["Date"])
consumer_expectations_df


#### Consumer Economic Conditions checks

In [ ]:
#### first set of checks
consumer_economic_conditions_df.sample(5)         # Date is not the index - np
consumer_economic_conditions_df.head()            # Starts at the correct date
consumer_economic_conditions_df.tail()            # ends at 2026-02
consumer_economic_conditions_df.columns           # No spaces which is good but do need to rename  + No need to drop columns
consumer_economic_conditions_df.dtypes            # Npo changes needed

    #### Initial adjustments
consumer_economic_conditions_df.rename(columns={"Unnamed: 0":"Date","3mma":"Economic_Conditions_index"}, inplace=True)     # Fixed the index


#### Second set of checks
consumer_economic_conditions_df.columns           #Need to drop the recessions column
consumer_economic_conditions_df.duplicated(subset="Date", keep=False).value_counts()      # All are false so no duplicated values
consumer_economic_conditions_df.isna().sum()                                              # No missing values for any of the dates+ missing for recession + Need to check if all the 12 months are there

    # Adjustments
consumer_economic_conditions_df.drop(columns=["Recession"], inplace=True)       # Remove the Recession column
consumer_economic_conditions_df["Date"]= pd.to_datetime(consumer_economic_conditions_df["Date"])



        # Checks to make sure that all the dates are there for the period from 2000 to 2026.
print(pd.date_range(start= "2000-01-01", end = "2026-01-01", freq="MS").difference(consumer_economic_conditions_df["Date"]))
                    # Sets up a date range, looks at the month start, then compares and then prints the ones that are different
                    ## There are no missing values
consumer_economic_conditions_df

#### Personal Consumption Expenditures - Quarterly - Not adjusted

In [ ]:
#### Initial checks
personal_consumption_exp_unadjusted.sample(5)

### Main Clean dataset with all variables
---

In [ ]:
#### Monthly Data

initial_df = pd.merge(consumer_economic_conditions_df, consumer_expectations_df, on="Date", how="inner")
clean_df1= pd.merge(initial_df,consumer_sentiment_df, on="Date", how="outer")
clean_df_monthly = pd.merge(clean_df1, percent_change_monthly_adjusted, on ="Date" , how = "outer")



clean_df_monthly.head(5)

clean_df_monthly.rename(columns= {"Economic_Conditions_index":"Economic_Conditions_Index",
                          "Consumer_Expectations_index":"Consumer_Expectations_Index",
                          "Sentiment_index":"Sentiment_Index",
                          "Percentage_Change_Consumption": "Percentage _Change_Consumption_Monthly",
                          "Consumption_Expenditures( Billions)": "Consumption_Expenditures",
                          "Percentage_Change_Consumption_Yearly":"Percentage_Change_Consumption_Yearly"}, inplace=True)
clean_df_monthly

In [ ]:
#### Yearly Data


clean_df_yearly= pd.merge(percent_change_yearly_no_adjusted, real_consumption_expenditures_annual, on ="Date" , how="outer")

clean_df_yearly.rename(columns= {
                          "Consumption_Expenditures( Billions)": "Consumption_Expenditures_Yearly",
                          "Percentage_Change_Consumption_Yearly":"Percentage_Change_Consumption_Yearly"}, inplace=True)

clean_df_yearly


### Main Clean dataset with all variables( Z score)
---

In [ ]:
## Monthly data Zscores
zscore_clean_df_monthly = pd.DataFrame({
    "Date": clean_df_monthly["Date"],
    "Economic_Conditions_Index": stats.zscore(clean_df_monthly["Economic_Conditions_Index"], nan_policy='omit'),
    "Consumer_Expectations_Index": stats.zscore(clean_df_monthly["Consumer_Expectations_Index"], nan_policy='omit'),
    "Sentiment_Index": stats.zscore(clean_df_monthly["Sentiment_Index"], nan_policy='omit'),
    "Percentage _Change_Consumption_Monthly": stats.zscore(clean_df_monthly["Percentage _Change_Consumption_Monthly"], nan_policy='omit' ),
})

zscore_clean_df_monthly

In [ ]:
### Z score for yearly Variables

zscore_clean_df_yearly = pd.DataFrame({
    "Date": clean_df_yearly["Date"],
    "Percentage_Change_Consumption_Yearly": stats.zscore(clean_df_yearly["Percentage_Change_Consumption_Yearly"], nan_policy='omit'),
    "Consumption_Expenditures_Yearly": stats.zscore(clean_df_yearly["Consumption_Expenditures_Yearly"], nan_policy='omit'),

})

zscore_clean_df_yearly


### Descriptive statistics
---

In [ ]:
clean_df_monthly.describe()

In [ ]:
zscore_clean_df_monthly.describe()

In [ ]:
clean_df_yearly

In [ ]:
zscore_clean_df_yearly

### Visualisation
---

##### Overall Sentiment to Overall Consumption

In [ ]:
## Plot
    ## Initial features
plt.figure(figsize=(14,8))
plt.xticks(fontsize=14, rotation = 90)
plt.yticks(fontsize=14)

ax1 = plt.gca()
ax2 = ax1.twinx()
    ## Plots
ax1.plot(clean_df["Date"],
         clean_df["Sentiment_Index"],
         color="Blue",
         linewidth=2,
         label="Sentiment_index")

ax2.plot(real_consumption_expenditures_annual["Date"],
         real_consumption_expenditures_annual["Consumption_Expenditures"],
         color="Red",
         linewidth=2,
         label="Consumption_Expenditures( Billions)")

    ## Adding in the date functionalities
Year = mdates.YearLocator()
Months = mdates.MonthLocator()
Years_FMT = mdates.DateFormatter('%Y')

ax1.xaxis.set_major_locator(Year)
ax1.xaxis.set_minor_locator(Months)
ax1.xaxis.set_major_formatter(Years_FMT)


    ## Final features

ax1.set_xlabel("Date")
ax1.set_ylabel("Sentiment Index")
ax2.set_ylabel("Consumption Expenditures( Billions)")
plt.title("Consumer Sentiment Index 2000-2026", fontsize=16)
plt.xlim(pd.Timestamp("2000-01-01"), pd.Timestamp("2026-01-01"))
fig = plt.gcf()
fig.legend()
plt.show()


Consumption is steadily rising compared to sentiment -- so the variables seem decoupled. Try the percent changes.


What about growth rates of consumption ?


##### Sentiment to Percentage change in consumption( Z Scores)

In [ ]:
aligned_df = zscore_clean_df[["Date", "Sentiment_index", "Percentage_Change_Monthly"]].dropna()
## Plot

plt.figure(figsize=(14,8))


plt.subplot(2, 1, 1)
plt.plot(zscore_clean_df["Date"],
         zscore_clean_df["Sentiment_index"],
         color="Blue",
         linewidth=2,
         label="Sentiment_index")
plt.title('Sentiment Z- Scores')
plt.xlabel('Date')
plt.ylabel('Z-Score')


plt.subplots_adjust(hspace=0.4)  # vertical gap (height)


plt.subplot(2, 1, 2)
plt.plot(aligned_df["Date"],
         aligned_df["Percentage_Change_Monthly"],
         color="Blue",
         linewidth=2,
         label="Sentiment_index")
plt.title('Consumption Z-Score')
plt.xlabel('Date')
plt.ylabel('Consumption Z-Score')


##### Percentage change in consumption onto Consumer Economic Conditions + Consumer Expectations( Monthly consumption data)

In [ ]:
aligned_df = zscore_clean_df[["Date","Economic_Conditions_index", "Consumer_Expectations_index"	, "Sentiment_index", "Percentage_Change_Monthly"]].dropna()
## Plot

plt.figure(figsize=(14,8))


plt.subplot(2, 2, 1)
plt.plot(aligned_df["Date"],
         aligned_df["Economic_Conditions_index"],
         color="Blue",
         linewidth=2,
         label="Economic Conditions Z score")
plt.title('Economic Conditions Z- Scores')
plt.xlabel('Date')
plt.ylabel('Z-Score')

plt.subplot(2, 2, 2)
plt.plot(aligned_df["Date"],
         aligned_df["Consumer_Expectations_index"],
         color="Blue",
         linewidth=2,
         label="Consumer Expectations Z score")
plt.title('Consumer Expectations Z- Scores')
plt.xlabel('Date')
plt.ylabel('Z-Score')


plt.subplots_adjust(hspace=0.4)  # vertical gap (height)


plt.subplot(2, 2, 3)
plt.plot(aligned_df["Date"],
         aligned_df["Percentage_Change_Monthly"],
         color="Blue",
         linewidth=2,
         label="Sentiment_index")
plt.title('Consumption Z-Score')
plt.xlabel('Date')
plt.ylabel('Consumption Z-Score')

plt.subplot(2, 2, 4)
plt.plot(aligned_df["Date"],
         aligned_df["Percentage_Change_Monthly"],
         color="Blue",
         linewidth=2,
         label="Sentiment_index")
plt.title('Consumption Z-Score')
plt.xlabel('Date')
plt.ylabel('Consumption Z-Score')

##### Percentage change in consumption onto Consumer Economic Conditions + Consumer Expectations( Yearly consumption data)

In [ ]:
aligned_df = zscore_clean_df[["Date","Economic_Conditions_index", "Consumer_Expectations_index"	, "Sentiment_index","Percentage_Change_Monthly", "Percentage_change_Yearly"]].dropna()
## Plot

plt.figure(figsize=(14,8))


plt.subplot(2, 2, 1)
plt.plot(aligned_df["Date"],
         aligned_df["Economic_Conditions_index"],
         color="Blue",
         linewidth=2,
         label="Economic Conditions Z score")
plt.title('Economic Conditions Z- Scores')
plt.xlabel('Date')
plt.ylabel('Z-Score')

plt.subplot(2, 2, 2)
plt.plot(aligned_df["Date"],
         aligned_df["Consumer_Expectations_index"],
         color="Blue",
         linewidth=2,
         label="Consumer Expectations Z score")
plt.title('Consumer Expectations Z- Scores')
plt.xlabel('Date')
plt.ylabel('Z-Score')


plt.subplots_adjust(hspace=0.4)  # vertical gap (height)


plt.subplot(2, 2, 3)
plt.plot(aligned_df["Date"],
         aligned_df["Percentage_change_Yearly"],
         color="Blue",
         linewidth=2,
         label="Sentiment_index")
plt.title('Consumption Z-Score( Yearly)')
plt.xlabel('Date')
plt.ylabel('Consumption Z-Score( Yearly)')

plt.subplot(2, 2, 4)
plt.plot(aligned_df["Date"],
         aligned_df["Percentage_change_Yearly"],
         color="Blue",
         linewidth=2,
         label="Sentiment_index")
plt.title('Consumption Z-Score( Yearly)')
plt.xlabel('Date')
plt.ylabel('Consumption Z-Score( Yearly)')

##### Consumer Economic Conditions vs Consumer Expectations

In [ ]:
## Plot
plt.figure(figsize=(16,10))
plt.xticks(fontsize=14, rotation = 90)
plt.yticks(fontsize=14)

ax1 = plt.gca()
ax2 = ax1.twinx()

ax1.plot(clean_df["Date"],
         clean_df["Economic_Conditions_index"],
         color="Blue",
         linewidth=2,
         label="Economic Conditions")
ax2.plot(clean_df["Date"],
         clean_df["Consumer_Expectations_index"],
         color="Red",
         linewidth=2,
         label="Consumer Expectations")

    ## Adding in the date functionalities
Year = mdates.YearLocator()
Months = mdates.MonthLocator()
Years_FMT = mdates.DateFormatter('%Y')

ax1.xaxis.set_major_locator(Year)
ax1.xaxis.set_minor_locator(Months)
ax1.xaxis.set_major_formatter(Years_FMT)





ax1.set_xlabel("Date")
ax1.set_ylabel("Economic Conditions")
ax2.set_ylabel("Consumer Expectations")
plt.xlim(pd.Timestamp("2000-01-01"), pd.Timestamp("2026-01-01"))

plt.title("Consumer Economic Conditions vs Consumer Expectations", fontsize=16)
fig= plt.gcf()
fig.legend()
fig.show()